# 18 - What sets the 0.5 m floor in the 125 m delivery error? (Kaggle)

Notebook 17 showed that with perfect burns the median delivery error from 125 m is still 0.50 m. The suspect is the **truth-only unmodelled acceleration** (an assumed constant bias of 1e-6 m/s^2 per axis, 1 sigma, that the filter does not know about), which over the roughly 790 s arc alone moves the spacecraft about 0.3 m per axis. This notebook removes it: the same seeds 0-49 from 125 m, no detector, with (a) **no unmodelled acceleration, normal burn errors** and (b) **no unmodelled acceleration and perfect burns**, which leaves only navigation and everything else. With notebooks 16 and 17 that makes a 2 by 2 table (acceleration on/off x burn errors on/off) on the same seeds.

Settings: **GPU T4 x2**, **Internet on**. No dataset needed. Save Version, Save & Run All (Commit). Two flight processes side by side (50 landings each); about 3 hours.

In [ ]:
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /tmp/touch_down || (cd /tmp/touch_down && git pull -q)
!sh /tmp/touch_down/scripts/kaggle_setup.sh 2>&1 | tail -5

In [ ]:
# the 195 MB global model, then the wide terrain (writes /tmp/data/nightingale_wide_5cm.npz, about 15 s)
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python scripts/fetch_data.py --global
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python scripts/build_wide_dtm.py --out-dir /tmp/data

In [ ]:
import subprocess
W = '/kaggle/working/accel_sens'
run = 'python /tmp/touch_down/scripts/kaggle_run.py --max-hours 6 --wide --seeds 0 50 --accel-scale 0.0'
cases = {'accel0_burn1': '', 'accel0_burn0': '--burn-error-scale 0.0'}
procs = [subprocess.Popen(f"{run} {extra} --out {W}/{name}", shell=True) for name, extra in cases.items()]
for p in procs:
    p.wait()
print('both flight processes finished', [p.returncode for p in procs])